# DEMO MODEL

# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Rescaling
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model

# Prepare data frame of file names and legibility determinations

In [ ]:
#read in csv of signatures as a dataframe
df = pd.read_csv('/content/drive/My Drive/signatures project/Final Assignments_UTF.csv')

In [ ]:
df.head(20)

,File Name,Signature Text,Legibility Rating (1–5),Comments / Notes,File Name.1,Anthony - Legible or Illegible,Assigned Number,Anthony - Type Name,Average,Final Assignment
0,sig001.jpg,Ruma Jy,3,Stylized cursive; last name abbreviated/unclear,sig001.jpg,Illegible,1,Rudy,2.0,Illegible
1,sig002.jpg,B. Navya,4,Clear initials and first name; good spacing,sig002.jpg,Legible,5,B. Navya,4.5,Legible
2,sig003.jpg,V. Jyothi,4,Initial with clear full first name,sig003.jpg,Legible,5,V. Jyothi,4.5,Legible
3,sig004.jpg,Jabeena,4,Readable single-name signature,sig004.jpg,Legible,5,Jabeena,4.5,Legible
4,sig005.jpg,Saniya,3,Loops and flourishes reduce clarity slightly,sig005.jpg,Legible,5,Saniya,4.0,Legible
5,sig006.jpg,G. Preethi,4,Clear initial and name; confident strokes,sig006.jpg,Legible,5,G. Preetti,4.5,Legible
6,sig007.jpg,M. Nikhitha,4,Well-formed letters; easy to distinguish,sig007.jpg,Legible,5,M. Nikhitha,4.5,Legible
7,sig008.jpg,Venkay,2,Highly stylized; letter forms ambiguous,sig008.jpg,Illegible,1,V Daky,1.5,Illegible
8,sig009.jpg,Kalyani,3,Readable but light strokes,sig009.jpg,Legible,5,Kelyoru,4.0,Legible
9,sig010.jpg,Bathyuska,2,Unusual letter connections lower readability,sig010.jpg,Legible,5,Prathyusha,3.5,Legible


In [ ]:
df = df[['File Name','Final Assignment']]

In [ ]:
df.shape

(686, 2)

# Retrieve files from file names in data frame

In [ ]:
def get_file_path(file_name):
  return '/content/drive/My Drive/signatures project/' + file_name

In [ ]:
#replace file name in data frame with entire file path
df['File Name'] = df['File Name'].apply(get_file_path)

In [ ]:
df.head()

,File Name,Final Assignment
0,/content/drive/My Drive/signatures project/sig...,Illegible
1,/content/drive/My Drive/signatures project/sig...,Legible
2,/content/drive/My Drive/signatures project/sig...,Legible
3,/content/drive/My Drive/signatures project/sig...,Legible
4,/content/drive/My Drive/signatures project/sig...,Legible


In [ ]:
df['Legibility'] = df['Final Assignment'].astype(str)

In [ ]:
df.head()

,File Name,Final Assignment,Legibility
0,/content/drive/My Drive/signatures project/sig...,Illegible,Illegible
1,/content/drive/My Drive/signatures project/sig...,Legible,Legible
2,/content/drive/My Drive/signatures project/sig...,Legible,Legible
3,/content/drive/My Drive/signatures project/sig...,Legible,Legible
4,/content/drive/My Drive/signatures project/sig...,Legible,Legible


In [ ]:
df.drop('Final Assignment', axis=1, inplace=True)

In [ ]:
df

,File Name,Legibility
0,/content/drive/My Drive/signatures project/sig...,Illegible
1,/content/drive/My Drive/signatures project/sig...,Legible
2,/content/drive/My Drive/signatures project/sig...,Legible
3,/content/drive/My Drive/signatures project/sig...,Legible
4,/content/drive/My Drive/signatures project/sig...,Legible
...,...,...
681,/content/drive/My Drive/signatures project/sig...,Legible
682,/content/drive/My Drive/signatures project/sig...,Illegible
683,/content/drive/My Drive/signatures project/sig...,Legible
684,/content/drive/My Drive/signatures project/sig...,Legible


In [ ]:
df.groupby('Legibility').count()

,File Name
Legibility,
Illegible,410
Legible,276


# Set up model in Keras

In [ ]:
#split data frame into training and test data
train_df, test_df = train_test_split(df, test_size=0.2)

In [ ]:
#normalize pixesl for network efficiency
datagen = ImageDataGenerator(rescale=1./225)

In [ ]:
#set dimensions to resize images
IMAGE_HEIGHT = 140
IMAGE_WIDTH = 205
BATCH_SIZE = 686

In [ ]:
# set up training data

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col = 'File Name',
    y_col = 'Legibility',
    batch_size = BATCH_SIZE,
    class_mode = 'binary',
    target_size = (IMAGE_HEIGHT, IMAGE_WIDTH),
    color_mode = 'grayscale'
)

# set up test data

test_generator = datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col = 'File Name',
    y_col = 'Legibility',
    batch_size = BATCH_SIZE,
    class_mode = 'binary',
    target_size = (IMAGE_HEIGHT, IMAGE_WIDTH),
    color_mode = 'grayscale'
)

Found 548 validated image filenames belonging to 2 classes.
Found 138 validated image filenames belonging to 2 classes.


In [ ]:
#add class wieghts to balance out the model
balanced_weights = {
    0: 1.0,
    1: 1.5
}

In [ ]:
# set up CNN
model = Sequential([
    Rescaling(1./255, input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 1)),

    # first convolutional block
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Flatten the results to feed into a dense layer
    Flatten(),

    # Fully Connected Layer
    Dense(64, activation='relu'),
    Dropout(0.5), # Helps prevent overfitting

    # Output Layer: 1 neuron with a sigmoid activation for binary classification (0 or 1)
    Dense(1, activation='sigmoid')
])

#compile model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

#display network architecture summary
model.summary()

#train the model
EPOCHS = 15

history = model.fit(
    train_generator,
    validation_data = test_generator,
    epochs = EPOCHS
)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 140, 205, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 138, 203, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 69, 101, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 67, 99, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 33, 49, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 103488)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │     6,623,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,642,177 (25.34 MB)

 Trainable params: 6,642,177 (25.34 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33s/step - accuracy: 0.5657 - loss: 0.6931

1/1 ━━━━━━━━━━━━━━━━━━━━ 39s 39s/step - accuracy: 0.5657 - loss: 0.6931 - val_accuracy: 0.5870 - val_loss: 0.6925
Epoch 2/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 25s 25s/step - accuracy: 0.6004 - loss: 0.6924 - val_accuracy: 0.5870 - val_loss: 0.6918
Epoch 3/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 25s 25s/step - accuracy: 0.6004 - loss: 0.6916 - val_accuracy: 0.5870 - val_loss: 0.6907
Epoch 4/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 26s 26s/step - accuracy: 0.6004 - loss: 0.6903 - val_accuracy: 0.5870 - val_loss: 0.6895
Epoch 5/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 24s 24s/step - accuracy: 0.6004 - loss: 0.6890 - val_accuracy: 0.5870 - val_loss: 0.6880
Epoch 6/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 26s 26s/step - accuracy: 0.6004 - loss: 0.6872 - val_accuracy: 0.5870 - val_loss: 0.6862
Epoch 7/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 25s 25s/step - accuracy: 0.6004 - loss: 0.6852 - val_accuracy: 0.5870 - val_loss: 0.6844
Epoch 8/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 41s 41s/step - accuracy: 0.6004 - loss: 0.6821 - val_accuracy: 0.5870 - val_loss: 0.6825
Epoch 9/15


In [ ]:
model.save('/content/drive/MyDrive/signature_classifier.keras')